In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn import tree
import numpy as np

#데이터셋 불러오기
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 고윳값 Drop
X_train=train_df.drop(["index","name","health_status"],axis=1)
X_test=test_df.drop(["index","name"],axis=1)
y_train=train_df['health_status']

# 주요 피처 리스트
important_features = ['iron_lvl', 'stature_inch', 'fat_content', 'age_enzyme_ratio', 'Total_Cholesterol']

# 상호작용 피처 생성 함수
def create_interaction_features(df):
    
    # 추가 피처 생성 (주어진 코드대로)
    df['bmi'] = df['mass_kg'] / ((df['stature_inch'] + 100) / 100) ** 2
    df['age'] = 2024 - df['birth'] + 1
    df['bp'] = df['bp_high'] - df['bp_low']
    df['bp_tf'] = (df['bp_high'] > 130) & (df['bp_low'] > 85)
    df['bp_mean'] = (df['bp_high'] + df['bp_low']) / 2
    df['Waist_to_Height_Ratio'] = df['midsection_inch'] / df['stature_inch']
    df['LDL_to_HDL_Ratio'] = df['chol_bad'] / df['chol_good']
    df['Total_Cholesterol'] = df['chol_good'] + df['chol_bad']
    df['Enzyme_Avg'] = df[['enzyme_1', 'enzyme_2', 'enzyme_3']].mean(axis=1)
    df['Vision_Imbalance'] = (df['vision_l'] != df['vision_r'])
    df['vision_mean'] = (df['vision_l'] + df['vision_r']) / 2
    df['Metabolic_Syndrome'] = (
        (df['bp_high'] > 130) |
        (df['bp_low'] > 85) |
        (df['chol_bad'] > 100) |
        (df['midsection_inch'] > 35) |
        (df['sugar_lvl'] > 100)
    )
    df['waist_to_age_ratio'] = df['midsection_inch'] / df['age']
    df['audio_problem'] = ((df['audio_l'] != 'N') | (df['audio_r'] != 'N')).astype(int)
    df['urine_marker_flag'] = (df['urine_marker'] > 0).astype(int)
    df['age_enzyme_ratio'] = df['Enzyme_Avg'] / df['age']
    # 기본적인 상호작용 피처들
    df['iron_fat'] = df['iron_lvl'] * df['fat_content']
    df['stature_age'] = df['stature_inch'] * df['age_enzyme_ratio']
    df['cholesterol_iron'] = df['Total_Cholesterol'] * df['iron_lvl']

    # 나눗셈 상호작용
    df['iron_per_fat'] = df['iron_lvl'] / (df['fat_content'] + 1e-6)
    df['stature_per_cholesterol'] = df['stature_inch'] / (df['Total_Cholesterol'] + 1e-6)
    df['age_per_iron'] = df['age_enzyme_ratio'] / (df['iron_lvl'] + 1e-6)

    # 덧셈 상호작용
    df['iron_plus_fat'] = df['iron_lvl'] + df['fat_content']
    df['age_plus_cholesterol'] = df['age_enzyme_ratio'] + df['Total_Cholesterol']

    # 뺄셈 상호작용
    df['iron_minus_fat'] = df['iron_lvl'] - df['fat_content']
    df['cholesterol_minus_stature'] = df['Total_Cholesterol'] - df['stature_inch']

    # 비선형 상호작용 (제곱, 루트)
    df['iron_squared'] = df['iron_lvl'] ** 2
    df['sqrt_cholesterol'] = df['Total_Cholesterol'] ** 0.5
    
    # 곱셈 기반 상호작용
    df['iron_stature'] = df['iron_lvl'] * df['stature_inch']
    df['age_iron_squared'] = df['age_enzyme_ratio'] * df['iron_squared']
    df['cholesterol_iron'] = df['Total_Cholesterol'] * df['iron_lvl']
    df['bmi_iron'] = df['bmi'] * df['iron_lvl']

    # 나눗셈 기반 상호작용
    df['iron_per_stature'] = df['iron_lvl'] / (df['stature_inch'] + 1e-6)
    df['cholesterol_per_age'] = df['Total_Cholesterol'] / (df['age_enzyme_ratio'] + 1e-6)
    df['stature_per_bmi'] = df['stature_inch'] / (df['bmi'] + 1e-6)

    # 덧셈 및 뺄셈 기반 상호작용
    df['iron_plus_cholesterol'] = df['iron_lvl'] + df['Total_Cholesterol']
    df['stature_minus_mass'] = df['stature_inch'] - df['mass_kg']
    df['cholesterol_minus_age'] = df['Total_Cholesterol'] - df['age_enzyme_ratio']

    # 비선형 상호작용
    df['sqrt_iron_cholesterol'] = np.sqrt(df['iron_lvl'] * df['Total_Cholesterol'])
    df['stature_squared'] = df['stature_inch'] ** 2
    df['log_iron_lvl'] = np.log1p(df['iron_lvl'])
    
    return df

# X_train, X_test 데이터에 적용
X_train = create_interaction_features(X_train)
X_test = create_interaction_features(X_test)

# 결과 확인
X_test


In [ ]:
drop=[
 'audio_l',
 'audio_r',
 'bp_tf',
 'iron_squared',
 'log_iron_lvl',
 'sqrt_cholesterol',
 'sqrt_iron_cholesterol',
 'stature_squared',
 'urine_marker_flag',
 'Metabolic_Syndrome',
 'Vision_Imbalance',
 'age',
 'Total_Cholesterol',
 'urine_marker',
 'audio_problem',
 'mass_kg',
 'oral_issues']
X_train=X_train.drop(drop,axis=1)
X_test=X_test.drop(drop,axis=1)


In [ ]:
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
import numpy as np
import os

# XGBoost 모델 정의
xgb_model = xgb.XGBClassifier()

# 랜덤 서치에서 탐색할 하이퍼파라미터 범위 정의
param_dist = {
    'subsample': [0.9, 0.95,1.0],  # 기존의 0.9에서 근처 범위로 유지
    'reg_lambda': [0.25, 0.3, 0.35],  # 0.3 기준으로 조금 더 탐색
    'reg_alpha': [0.01,0.02,0.03,0.04],  # 0.05 기준으로 근처 범위 정렬
    'n_estimators': [300,350,400,500,600],  # 기존의 190을 중심으로 조금 넓힘
    'min_child_weight': [5,6,5.5,7,6.5],  # 기존의 4에서 범위를 조금 넓힘
    'max_depth': [3,4,5,6,7,8,9],  # 중복된 10 제거, 7을 기준으로 더 깊거나 얕게 설정
    'learning_rate': [0.05,0.06,0.07,0.08,0.09],  # 0.07을 기준으로 조금 넓힘, 0.03 앞으로 이동
    'gamma': [0.1, 0.15, 0.2],  # 0.2를 기준으로 근처 탐색 정렬
    'colsample_bytree': [0.7, 0.75, 0.8,0.85,0.9]  # 0.7을 기준으로 범위를 설정, 0.6 앞으로 이동
}
# RandomizedSearchCV로 하이퍼파라미터 최적화 수행
random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_dist,
    n_iter=30,  # 더 적은 조합으로 탐색
    scoring='roc_auc',  # roc_auc 스코어로 평가
    cv=3,  # 교차 검증 fold 수
    verbose=3,
    random_state=42,
    n_jobs=-1
)

# 랜덤 서치 실행
random_search.fit(X_train, y_train)

# 최적의 파라미터와 모델 확인
best_params = random_search.best_params_
best_model = random_search.best_estimator_

# 결과 파일에 저장
# 파일 이름 정의
filename = 'xg_boost1.0.txt'

# 파일에 결과 작성 (파일이 없으면 생성)
cv_results = random_search.cv_results_
with open(filename, 'a' if os.path.exists(filename) else 'x') as f:
    for mean_score, params in zip(cv_results['mean_test_score'], cv_results['params']):
        f.write(f"Params: {params}, Mean CV Score: {mean_score:.3f}\n")
    
    f.write("\nBest Parameters:\n")
    f.write(f"{best_params}\n")
    f.write(f"Best Score: {random_search.best_score_:.3f}\n")
    f.write("-------------------------------------------------------------------------------------------------------------------------------------------------------------------------\n")


In [ ]:
import numpy as np  # numpy가 import되어 있는지 확인


def ensemble_predict(xgb_probs, lgb_probs):
    # 두 모델의 확률을 기준으로 결합
    ensemble_probs = []
    
    for xgb_prob, lgb_prob in zip(xgb_probs, lgb_probs):
        if xgb_prob < 0.5 and lgb_prob < 0.5:
            ensemble_probs.append(min(xgb_prob, lgb_prob))
        elif xgb_prob >= 0.5 and lgb_prob >= 0.5:
            ensemble_probs.append(max(xgb_prob, lgb_prob))
        else:
            ensemble_probs.append(max(xgb_prob, lgb_prob))  # 하나만 0.5 이상일 경우
            
    return np.array(ensemble_probs)

# 앙상블 예측 수행
ensemble_probs = ensemble_predict(xgb_probs, lgb_probs)

# 앙상블 확률 출력
print('best [0.53321308 0.02115297 0.03852971 ... 0.06604431 0.12076061 0.83912494]')
print("Ensemble Probabilities:", ensemble_probs)
ㅋ

In [ ]:
# 제출 파일 저장
submission_df = pd.read_csv('sample_submission.csv')
submission_df.health_status = ensemble_prob
submission_df.to_csv('submission.csv', index=False)